In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

from scipy.io import loadmat, savemat

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models

In [ ]:
DATA_PATH = os.path.join("..", "..", "content", "mnist-original.mat")
MODEL_PATH = os.path.join("..", "..", "models", "resnet18_mnist_surrogate_or.pth")  

In [4]:
mnist = loadmat(DATA_PATH)
mnist_data, mnist_label = np.array(mnist["data"].T), np.array(mnist["label"][0].T)

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [6]:
class MNISTDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx].reshape(28, 28).astype(np.uint8)
        if self.transform:
            img = self.transform(img)
        label = int(self.labels[idx])
        return img, label

In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

In [8]:
dataset = MNISTDataset(mnist_data, mnist_label, transform=transform)

In [9]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)

model.load_state_dict(torch.load(MODEL_PATH))

model = model.to(device)